# 16 — Graph RAG over the UMLS Knowledge Graph (with the AWMF Vector Guidelines)

**Objective.** Push the RAGAS metrics toward the 0.9 threshold expected of a clinical
decision-support system by adding a structured reasoning layer on top of the dense and
sparse retrieval already validated in the previous notebooks. That layer is a compact slice
of the UMLS knowledge graph, restricted to the ten general-practice diseases in scope.

**Why a slice, and not the whole ontology.** The complete UMLS release (the OWL export) is
roughly 40 GB and cannot be loaded inside a Colab session. The concept graph is reached
instead through the official UMLS Terminology Services (UTS) REST API. A one-time build
walks outward from the ten target diseases, collects the concepts directly connected to them
together with the relations between those concepts, records the German surface form of every
node, and caches the whole slice to Drive. Retrieval then reads this cached subgraph locally,
so there is no large file to host and no repeated network cost after the first build.

**Pipeline.** query → concept detection → one-hop neighbourhood in the cached subgraph →
graph-expanded German query plus explicit relation facts → dense (pgvector) + sparse (BM25)
→ Reciprocal Rank Fusion → cross-encoder rerank → grounded generation. Evaluation runs
against both the original MedQA ground truth and the corpus-grounded ground truth, so the
result is comparable with every earlier experiment.

**Prerequisites.** The same secrets as the earlier notebooks (`NEON_DATABASE_URL`,
`OPENROUTER_API_KEY`) plus `UMLS_API_KEY` from a free UTS account. The vector collection
`awmf_baseline_bge` and the two ground-truth files produced earlier are reused unchanged.

## Installs

In [1]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers sqlalchemy requests spacy rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 139.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 

## Vertex patch

A defensive shim so that importing parts of `langchain_community` does not pull in the
Vertex AI SDK, which is not installed in this environment.

In [2]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## Database, dense store, BM25 index, reranker, models

Identical infrastructure to notebook 15: the German AWMF chunks are served by pgvector for
dense retrieval and by an in-memory BM25 index for sparse retrieval, a cross-encoder handles
the final reranking, and Mistral-Large is the generator. The only prompt change is in the
answer template, which now instructs the model to combine the guideline passages with the
knowledge-graph facts.

In [4]:
import os, json, time, random
import pandas as pd, torch, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine, text
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv')

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30})

# 1. Dense vector store (read-only; collection built in the earlier notebooks)
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name="awmf_baseline_bge", connection=engine, use_jsonb=True)
dense_retriever = vs.as_retriever(search_kwargs={"k": 30})

# 2. Sparse BM25 index (built once from the same chunks)
print("Loading texts for BM25 index...")
with engine.connect() as conn:
    res = conn.execute(text("SELECT document FROM langchain_pg_embedding WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = 'awmf_baseline_bge')"))
    all_texts = [row[0] for row in res]
tokenized_corpus = [doc.lower().split(" ") for doc in all_texts]
bm25_retriever = BM25Okapi(tokenized_corpus)
print(f"BM25 index built with {len(all_texts)} chunks.")

# 3. Cross-encoder reranker
print("Loading reranker...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# Translation + HyDE, exactly as in the hybrid baseline
query_gen_prompt = PromptTemplate(template="""You are a German medical expert.
1. First, precisely TRANSLATE the English question into German.
2. Second, write a 2-sentence hypothetical answer to the question in formal German clinical guideline terminology.
Do NOT use bullet points. Output ONLY the German translation followed directly by the German hypothetical answer.

English Question:
{question}""", input_variables=["question"])

# Grounded, graph-aware answer prompt
qa_prompt = PromptTemplate(template="""You are an expert clinical assistant.
Answer the medical question in ENGLISH using the material below.

MATERIAL:
- German AWMF guideline passages provide the clinical detail.
- UMLS knowledge-graph facts state verified relations between concepts (drugs, findings, causes, comorbidities).

INSTRUCTIONS:
1. Ground the answer in the German guideline passages first.
2. Use the knowledge-graph facts to connect concepts the passages mention only indirectly.
3. Keep the answer precise and concise. Do not invent citations.

Context:
{context}

Question (English):
{question}

Answer (English):""", input_variables=["context","question"])
print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Setup complete.


## UMLS REST helpers: concept search, German atoms, and relations

Three thin wrappers over the UTS API. `search_cui` maps an English phrase to its Concept
Unique Identifier (CUI); `german_atoms` returns the German names attached to a concept, which
is what lets an English query reach the German corpus; and `concept_relations` — new in this
notebook — returns the edges of the knowledge graph around a concept. Every call is cached so
the one-time build never repeats a request.

In [5]:
import requests, functools, spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download as _dl; _dl("en_core_web_sm"); nlp = spacy.load("en_core_web_sm")

UMLS_API_KEY = userdata.get('UMLS_API_KEY')
UTS = "https://uts-ws.nlm.nih.gov/rest"
GENERIC = {"patient","patients","treatment","diagnosis","management","therapy","risk","cause",
           "symptom","disease","condition","prognosis"}

# Clinically meaningful predicates to keep; everything else is discarded to stop the graph
# from ballooning with administrative or purely lexical links.
KEEP_REL = {"may_treat","may_prevent","has_finding_site","associated_with","cause_of",
            "has_causative_agent","clinical_course_of","has_manifestation","manifestation_of",
            "disease_has_finding","disease_may_have_finding","has_symptom","symptom_of",
            "contraindicated_drug","induces","occurs_in","has_associated_morphology",
            "has_risk_factor","risk_factor_of","related_to"}
COARSE_REL = {"RO","RN","RB","PAR","CHD","SY"}  # broad fallback when no fine label is given

def candidate_phrases(question, max_n=6):
    doc = nlp(question)
    out, seen = [], set()
    for ch in doc.noun_chunks:
        t = " ".join(w.text for w in ch if not w.is_stop and not w.is_punct).strip()
        tl = t.lower()
        if len(tl) < 3 or tl in GENERIC or tl in seen: continue
        seen.add(tl); out.append(t)
        if len(out) >= max_n: break
    return out

@functools.lru_cache(maxsize=8192)
def search_cui(term):
    try:
        r = requests.get(f"{UTS}/search/current", params={"string": term, "apiKey": UMLS_API_KEY, "pageSize": 1, "searchType": "words"}, timeout=20)
        hits = r.json().get("result", {}).get("results", []) if r.ok else []
        if hits and hits[0].get("ui") not in (None, "NONE"): return hits[0]["ui"], hits[0].get("name", term)
    except: pass
    return None

@functools.lru_cache(maxsize=8192)
def german_atoms(cui):
    try:
        r = requests.get(f"{UTS}/content/current/CUI/{cui}/atoms", params={"language": "GER", "apiKey": UMLS_API_KEY, "pageSize": 50}, timeout=20)
        if not r.ok: return []
        names = []
        for a in r.json().get("result", []):
            n = (a.get("name") or "").strip()
            if n and all(n.lower() != x.lower() for x in names): names.append(n)
        return names[:4]
    except: return []

def _cui_from_url(u):
    return u.rstrip('/').split('/')[-1] if u else None

@functools.lru_cache(maxsize=8192)
def concept_relations(cui, pages=2, page_size=100):
    edges = []
    for p in range(1, pages+1):
        try:
            r = requests.get(f"{UTS}/content/current/CUI/{cui}/relations",
                             params={"apiKey": UMLS_API_KEY, "pageSize": page_size, "pageNumber": p}, timeout=25)
            if not r.ok: break
            rows = r.json().get("result", [])
            if not rows: break
            for rel in rows:
                fine = (rel.get("additionalRelationLabel") or "").strip()
                coarse = (rel.get("relationLabel") or "").strip()
                if fine and fine not in KEEP_REL and coarse not in COARSE_REL:
                    continue
                if not fine and coarse not in COARSE_REL:
                    continue
                tgt = _cui_from_url(rel.get("relatedId"))
                if not tgt or tgt == cui: continue
                edges.append((fine or coarse, tgt, (rel.get("relatedIdName") or "").strip()))
            if len(rows) < page_size: break
            time.sleep(0.05)
        except: break
    # de-duplicate on (predicate, target)
    uniq, seen = [], set()
    for e in edges:
        key = (e[0], e[1])
        if key in seen: continue
        seen.add(key); uniq.append(e)
    return uniq
print("UMLS helpers ready.")

UMLS helpers ready.


## One-time build of the disease-centred subgraph

The build starts from the ten target diseases. For each disease Mistral proposes the handful
of most clinically connected terms — typical symptoms, first-line drugs, common
complications, key comorbidities — which become additional seed concepts. Every seed is
resolved to a CUI, its one-hop relations are pulled from UTS, and both the seeds and their
neighbours are stored as graph nodes with their German names. The result is written to Drive
and re-loaded on every later run, so this cell is the only place that spends time on the API.
The disease list is read from the evaluation set itself so the graph stays perfectly aligned
with the benchmark.

In [6]:
GRAPH_FILE = DRIVE_PATH + "UMLS_GraphRAG_Subgraph.json"

# The ten GP diseases, taken from the evaluation set to stay aligned with the benchmark.
if 'disease' in df.columns and df['disease'].notna().any():
    DISEASES = sorted(df['disease'].dropna().unique().tolist())
else:
    DISEASES = ["Hypertension","Type 2 Diabetes","Asthma","Depression","Chronic Back Pain",
                "Coronary Heart Disease","Osteoarthritis","COPD","Dementia","Heart Failure"]
print("Target diseases:", DISEASES)

expand_prompt = PromptTemplate(template="""List the 10 most clinically connected terms for "{disease}" in general practice:
its typical symptoms, first-line drug treatments, common complications, and key comorbidities.
Return one term per line. No numbering, no explanations, no headings.""", input_variables=["disease"])

def expand_terms(disease):
    raw = safe_invoke(mistral, expand_prompt.format(disease=disease))
    terms = [t.strip(" -*\t").strip() for t in raw.replace(";", "\n").split("\n")]
    return [t for t in terms if 2 < len(t) < 60][:12]

def build_subgraph(force=False):
    if os.path.exists(GRAPH_FILE) and not force:
        g = json.load(open(GRAPH_FILE))
        print(f"Loaded cached subgraph: {len(g['nodes'])} nodes, {len(g['edges'])} edges")
        return g

    # 1. Seed terms = the diseases plus their LLM-proposed related concepts.
    seeds = []
    for d in DISEASES:
        seeds.append(d)
        try: seeds.extend(expand_terms(d))
        except Exception as e: print("  expand failed:", d, str(e)[:60])
    seeds = list(dict.fromkeys([s for s in seeds if s]))
    print(f"{len(seeds)} seed terms after expansion")

    # 2. Resolve seeds to concepts.
    nodes, seed_cuis = {}, []
    for term in seeds:
        hit = search_cui(term)
        if not hit: continue
        cui, name = hit
        if cui not in nodes:
            nodes[cui] = {"name": name, "de": german_atoms(cui), "seed": True}
            seed_cuis.append(cui)
    print(f"{len(seed_cuis)} seed concepts resolved")

    # 3. Pull one-hop relations for every seed concept; add neighbours as nodes.
    edges = []
    for i, cui in enumerate(seed_cuis):
        for pred, tgt, tname in concept_relations(cui):
            edges.append({"s": cui, "r": pred, "o": tgt, "o_name": tname})
            if tgt not in nodes:
                nodes[tgt] = {"name": tname, "de": german_atoms(tgt), "seed": False}
        if (i+1) % 10 == 0:
            print(f"  expanded {i+1}/{len(seed_cuis)} concepts -> {len(nodes)} nodes, {len(edges)} edges")
            json.dump({"nodes": nodes, "edges": edges}, open(GRAPH_FILE, "w"))  # checkpoint

    g = {"nodes": nodes, "edges": edges}
    json.dump(g, open(GRAPH_FILE, "w"))
    print(f"Subgraph built and cached -> {len(nodes)} nodes, {len(edges)} edges")
    return g

GRAPH = build_subgraph()

# Adjacency list and node table for fast lookup at query time.
NODES = GRAPH["nodes"]
ADJ = {}
for e in GRAPH["edges"]:
    ADJ.setdefault(e["s"], []).append((e["r"], e["o"], e["o_name"]))
print(f"Graph ready: {len(NODES)} nodes, {sum(len(v) for v in ADJ.values())} directed edges.")

Target diseases: ['Hypertension', 'Type 2 Diabetes', 'Asthma', 'Depression', 'Chronic Back Pain', 'Coronary Heart Disease', 'Osteoarthritis', 'COPD', 'Dementia', 'Heart Failure']
98 seed terms after expansion
90 seed concepts resolved
  expanded 10/90 concepts -> 997 nodes, 938 edges
  expanded 20/90 concepts -> 1469 nodes, 1444 edges
  expanded 30/90 concepts -> 1895 nodes, 1914 edges
  expanded 40/90 concepts -> 2269 nodes, 2346 edges
  expanded 50/90 concepts -> 2538 nodes, 2640 edges
  expanded 60/90 concepts -> 3064 nodes, 3220 edges
  expanded 70/90 concepts -> 3674 nodes, 3879 edges
  expanded 80/90 concepts -> 3922 nodes, 4177 edges
  expanded 90/90 concepts -> 4230 nodes, 4531 edges
Subgraph built and cached -> 4230 nodes, 4531 edges
Graph ready: 4230 nodes, 4531 directed edges.


## Graph-aware hybrid retrieval

For a given question the concepts in it are detected and matched against the subgraph. Their
one-hop neighbours contribute two things: extra German search terms (which widen the dense
and sparse queries) and explicit relation facts such as *Herzinsuffizienz — may_treat — ACE-Hemmer*.
The German terms are folded into the HyDE query before dense, sparse, RRF, and reranking run
exactly as before. The relation facts are handed to the generator as an extra, clearly labelled
context block, so the answer can be grounded in the graph as well as in the guideline passages.

In [7]:
def graph_neighbourhood(question, max_seeds=6, max_neighbours=12):
    hit_cuis = []
    for ph in candidate_phrases(question):
        h = search_cui(ph)
        if h: hit_cuis.append(h[0])
    hit_cuis = list(dict.fromkeys(hit_cuis))[:max_seeds]

    de_terms, facts = [], []
    for cui in hit_cuis:
        node = NODES.get(cui)
        if node:
            de_terms.extend(node.get("de", []))
        src_name = (node or {}).get("name", cui)
        for (pred, tgt, tname) in ADJ.get(cui, [])[:max_neighbours]:
            de_terms.extend(NODES.get(tgt, {}).get("de", []))
            facts.append(f"{src_name} — {pred.replace('_',' ')} — {tname}")

    de_terms = list(dict.fromkeys([t for t in de_terms if t]))
    facts = list(dict.fromkeys(facts))[:15]
    return " ".join(de_terms), facts

def retrieve_graph_hybrid(question, k_final=10):
    # 1. Translation + HyDE
    hyde = safe_invoke(mistral, query_gen_prompt.format(question=question))

    # 2. Graph expansion: extra German terms + explicit relation facts
    graph_terms, graph_facts = graph_neighbourhood(question)

    # 3. Enriched search query
    hybrid_query = f"{hyde} {graph_terms}".strip()

    # 4. Dense (pgvector)
    dense_docs = [d.page_content for d in dense_retriever.invoke(hybrid_query)]

    # 5. Sparse (BM25)
    sparse_docs = bm25_retriever.get_top_n(hybrid_query.lower().split(" "), all_texts, n=30)

    # 6. Reciprocal Rank Fusion
    rrf = {}
    for rank, doc in enumerate(dense_docs): rrf[doc] = rrf.get(doc, 0.0) + 1.0/(60+rank)
    for rank, doc in enumerate(sparse_docs): rrf[doc] = rrf.get(doc, 0.0) + 1.0/(60+rank)
    fused = [doc for doc, _ in sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:30]]

    # 7. Cross-encoder rerank
    scores = reranker.predict([[hybrid_query, t] for t in fused])
    reranked = [t for _, t in sorted(zip(scores, fused), key=lambda x: x[0], reverse=True)][:k_final]

    # 8. Prepend the knowledge-graph facts as an extra context block
    contexts = list(reranked)
    if graph_facts:
        kg_block = "Wissensgraph-Fakten (UMLS):\n" + "\n".join("- " + f for f in graph_facts)
        contexts = [kg_block] + contexts
    return contexts, graph_facts

# Smoke test
_q = df.iloc[0]['English_Open_Question']
_ctx, _facts = retrieve_graph_hybrid(_q)
print("Question:", _q[:90])
print("Graph facts found:", len(_facts))
for f in _facts[:5]: print("   ", f)
print("Top retrieved passage:", _ctx[-1][:200], "...")

Question: A 58-year-old man comes to the emergency department because of increasing shortness of bre
Graph facts found: 0
Top retrieved passage: grunde. Klinisch kann sich diese durch eine Zunahme der Dyspnoe, des Hustens, des Sputumvolumens und/oder 
der Sputumpurulenz darstellen. [11] 
Vertiefende Informationen 
Eine Verschlechterung der res ...


## Generation loop

Every question is answered with the graph-aware pipeline. Results are written to Drive after
each item, so the run resumes cleanly if the session drops. Both ground truths are stored
alongside each answer so the evaluation cell can score against either without regenerating.

In [8]:
# Corpus-grounded ground truth (built in notebook 11); MedQA ground truth is in the dataset.
gt_df = pd.read_csv(DRIVE_PATH + "AWMF_CorpusGrounded_GroundTruth.csv")
gt_map = gt_df.set_index('English_Open_Question')['corpus_ground_truth'].to_dict()

work = df.reset_index(drop=True)  # full 200 questions
print(f"Generating Graph-RAG answers for {len(work)} questions")

out_file = DRIVE_PATH + "GRAPHRAG_results.json"
if os.path.exists(out_file):
    res = json.load(open(out_file)); done = set(res["question"])
    print(f"  resuming -- {len(done)} already done")
else:
    res = {"question":[],"answer":[],"contexts":[],"graph_facts":[],
           "medqa_ground_truth":[],"corpus_ground_truth":[]}; done = set()

for i in range(len(work)):
    r = work.iloc[i]; q = r['English_Open_Question']
    if q in done: continue
    try:
        ctx, facts = retrieve_graph_hybrid(q)
        ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))

        res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
        res["graph_facts"].append(facts)
        res["medqa_ground_truth"].append(r['English_Correct_Text'])
        res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
        done.add(q)
        json.dump(res, open(out_file, "w"))

        if len(res["question"]) % 10 == 0: print(f"  [graph-rag] {len(res['question'])}/{len(work)}")
        time.sleep(1.5)
    except Exception as e:
        print("err", i, str(e)[:100]); time.sleep(8)

print(f"Graph-RAG generation complete -> {out_file}")

Generating Graph-RAG answers for 200 questions
  [graph-rag] 10/200
  [graph-rag] 20/200
  [graph-rag] 30/200
  [graph-rag] 40/200
  [graph-rag] 50/200
  [graph-rag] 60/200
  [graph-rag] 70/200
  [graph-rag] 80/200
  [graph-rag] 90/200
  [graph-rag] 100/200
  [graph-rag] 110/200
  [graph-rag] 120/200
  [graph-rag] 130/200
  [graph-rag] 140/200
  [graph-rag] 150/200
  [graph-rag] 160/200
  [graph-rag] 170/200
  [graph-rag] 180/200
  [graph-rag] 190/200
  [graph-rag] 200/200
Graph-RAG generation complete -> /content/drive/MyDrive/GRAPHRAG_results.json


## Evaluation against both ground truths

The same GPT-4o-mini judge and the same four RAGAS metrics as every earlier notebook, so the
numbers are directly comparable. The corpus-grounded score measures performance where an
answer genuinely exists in the AWMF guidelines; the MedQA score measures the full benchmark,
including the questions whose answers the graph relations help supply.

In [9]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score_graphrag(path, gt_key, only_in_corpus=False):
    d = json.load(open(path))
    idx = range(len(d["question"]))
    if only_in_corpus:
        idx = [i for i in idx if d["corpus_ground_truth"][i] not in (None, "", "NOT_IN_CORPUS")]
    dd = {"question":[d["question"][i] for i in idx],
          "answer":[d["answer"][i] for i in idx],
          "contexts":[d["contexts"][i] for i in idx],
          "ground_truth":[d[gt_key][i] for i in idx]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    return dict(out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3))

print("=== GRAPH RAG SCORES ===")
print("Corpus-Grounded (answerable subset):", score_graphrag(out_file, "corpus_ground_truth", only_in_corpus=True))
print("Full 200 (MedQA Ground Truth):      ", score_graphrag(out_file, "medqa_ground_truth"))

=== GRAPH RAG SCORES ===


/tmp/ipykernel_22043/1868756894.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_22043/1868756894.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_22043/1868756894.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_pre

Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

Corpus-Grounded (answerable subset): {'context_precision': np.float64(0.637), 'context_recall': np.float64(0.814), 'faithfulness': np.float64(0.685), 'answer_relevancy': np.float64(0.591)}


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

Full 200 (MedQA Ground Truth):       {'context_precision': np.float64(0.15), 'context_recall': np.float64(0.235), 'faithfulness': np.float64(0.483), 'answer_relevancy': np.float64(0.549)}
